# LUMEN-X: train + evaluate on Colab
Runtime > Change runtime type > **T4 GPU**. Run the cells in order. Replace the <PLACEHOLDERS>. Total time about 1 to 1.5 h.
Do NOT `pip install -r requirements.txt` here: it would replace Colab's GPU torch with a CPU build.

In [ ]:
!git clone <YOUR_REPO_URL> repo
%cd repo
!pip -q install lpips pyiqa tabulate scikit-image opencv-python-headless
import torch; print('GPU:', torch.cuda.is_available())

## 1. Data (LOL train/val/test split + unseen sets)

In [ ]:
!bash scripts/download_data.sh data
!python scripts/verify_data.py --data data
!ls data/LOL/our485/low | wc -l ; ls data/LOL/eval15/low | wc -l ; ls data/unseen

If the download fails (a mirror may be gone): get LOL from Kaggle ("LOL dataset") or the official Google Drive,
upload it so that `data/LOL/our485/{low,high}` and `data/LOL/eval15/{low,high}` exist, then run
`!python scripts/prepare_data.py --lol data/LOL --out data/splits.json`. Unseen sets (DICM, LIME, ExDark) can be any folder of dark photos in `data/unseen/<name>/`.

## 2. Smoke test (1 minute). Must finish without errors before the long runs

In [ ]:
!python train.py --data_root data/LOL --split_json data/splits.json --epochs 1 --limit 16 --bs 4 --workers 2 --out /content/smoke.pt
!python train.py --data_root data/LOL --split_json data/splits.json --epochs 1 --limit 16 --bs 4 --workers 2 --hybrid --out /content/smoke_h.pt

## 3. Train (best checkpoint chosen by VALIDATION PSNR)

In [ ]:
!python train.py --data_root data/LOL --split_json data/splits.json --epochs 150 --bs 8 --lr 3e-4 --out weights/zerodce_rgb.pt

In [ ]:
!python train.py --data_root data/LOL --split_json data/splits.json --epochs 150 --bs 8 --lr 3e-4 --hybrid --out weights/zerodce_hybrid.pt

In [ ]:
import pandas as pd
for n in ('rgb','hybrid'):
    d = pd.read_csv(f'weights/zerodce_{n}_log.csv'); print(n, 'best val PSNR', d.val_psnr.max().round(2), 'at epoch', int(d.loc[d.val_psnr.idxmax(),'epoch']))

If best val PSNR is low (below about 15) or the curve is flat, re-run the training cell with ONE change at a time
(e.g. `--nonref_scale 0.03`, `--l1_w 20`, `--lr 1e-3`, `--epochs 250`), writing to the same --out. Judge only by validation PSNR; never by the test set.

## 4. Final evaluation on CPU (the same device as the deployed app, so latency is honest)

In [ ]:
!CUDA_VISIBLE_DEVICES="" python evaluate.py --low_dir data/LOL/eval15/low --high_dir data/LOL/eval15/high --ablation --max_side 0 --out results/final/lol_eval15
!CUDA_VISIBLE_DEVICES="" python evaluate.py --low_dir data/unseen/DICM --out results/final/unseen_DICM
!CUDA_VISIBLE_DEVICES="" python evaluate.py --low_dir data/unseen/LIME --out results/final/unseen_LIME
!CUDA_VISIBLE_DEVICES="" python evaluate.py --low_dir data/unseen/ExDark100 --out results/final/unseen_ExDark100

In [ ]:
!python scripts/make_eval_md.py --lol results/final/lol_eval15 --unseen results/final/unseen_DICM results/final/unseen_LIME results/final/unseen_ExDark100 --usns <USN1,USN2,USN3,USN4> --team <T01> --prob P1 --out eval_P1_<T01>.md
!cat eval_P1_*.md

## 5. Download everything and commit it to the repo

In [ ]:
!zip -qr final_outputs.zip weights results/final eval_P1_*.md
from google.colab import files; files.download('final_outputs.zip')